# Inside the Encoder: Contextual Embeddings and Classifier Heads
**CPSC 471 · Applied AI** — companion notebook to the *Inside the Transformer* lecture

In this lab you will:
1. Meet **Hugging Face**, the place almost every open model lives
2. Watch BERT turn **"The bat flew"** into contextual embeddings
3. Prove that the vector for *bat* changes with its sentence
4. Put a small **fully connected network** on top of BERT to classify sentiment
5. See the same encoder do other jobs: fill in the blank, named entities, question answering, and search

**Runtime:** Google Colab works best. Parts 1–5 run on a CPU. Part 6 (fine-tuning all of BERT) needs a GPU: *Runtime → Change runtime type → T4 GPU*.

## Part 0 · Setup

In [ ]:
# Colab already has torch. This adds the Hugging Face libraries we use.
!pip install -q transformers datasets sentence-transformers scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("Using device:", device)

## Part 1 · Meet Hugging Face 🤗

[Hugging Face](https://huggingface.co) is to models what GitHub is to code. It has two halves:

| | What it is | Examples |
|---|---|---|
| **The Hub** (website) | A free host for models, datasets, and demo apps (*Spaces*) | `google-bert/bert-base-uncased`, the `stanfordnlp/sst2` dataset |
| **The libraries** (Python) | Code that downloads and runs what's on the Hub | `transformers`, `datasets`, `tokenizers`, `sentence-transformers` |

**Model IDs** look like `organization/model-name`. Every model has a **model card**: a README that says what it was trained on, what it's for, its license, and its limitations. *Read the card before you use a model.*

👉 Open <https://huggingface.co/models>, filter by the task **Text Classification**, and sort by *Most downloads*. Click one and skim its card.

### The fastest way in: `pipeline`
A `pipeline` bundles everything needed for a task into one call. The first run downloads the model (a few hundred MB) and caches it in `~/.cache/huggingface`.

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis",
                     model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

sentiment(["The movie was great",
           "The movie was a waste of two hours",
           "The bat flew"])

That's a finished sentiment classifier in three lines. **By the end of this notebook you will build one yourself.**

A pipeline hides three steps. We will open each of them:

```
text ──► tokenizer ──► model (encoder + head) ──► post-processing ──► label
```

In [ ]:
print(type(sentiment.tokenizer).__name__)
print(type(sentiment.model).__name__)
print(sentiment.model.config.id2label)

### The `Auto` classes
Below the pipeline you load pieces yourself with **`Auto` classes**. They read the model's config from the Hub and build the right architecture for you:

| Class | Gives you |
|---|---|
| `AutoTokenizer` | the tokenizer that matches the model |
| `AutoModel` | the bare encoder: outputs contextual embeddings |
| `AutoModelForSequenceClassification` | encoder + a classification head |
| `AutoModelForTokenClassification`, `AutoModelForQuestionAnswering`, `AutoModelForMaskedLM` | encoder + other heads |

Same `from_pretrained("org/model")` call for all of them. That's the whole trick.

In [ ]:
# You can also search the Hub from Python
from huggingface_hub import HfApi

for m in HfApi().list_models(pipeline_tag="fill-mask", sort="downloads", limit=5):
    print(m.id)

## Part 2 · Tokenizing "The bat flew"

In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL = "bert-base-uncased"          # short alias for google-bert/bert-base-uncased
tok = AutoTokenizer.from_pretrained(MODEL)

inputs = tok("The bat flew", return_tensors="pt")
print(inputs)
print(tok.convert_ids_to_tokens(inputs["input_ids"][0]))

- **`input_ids`**: each token's row number in BERT's vocabulary (about 30,000 entries)
- **`[CLS]`** (id 101) is added at the front and **`[SEP]`** (id 102) at the end
- **`attention_mask`**: 1 for real tokens, 0 for padding (padding appears when we batch sentences of different lengths)
- *uncased* means everything is lowercased first, so "The" becomes `the`

Rare words get split into **word pieces** (a `##` prefix means "continues the previous piece"):

In [ ]:
print(tok.tokenize("Tokenization of unbelievably rare words like pterodactyl"))

## Part 3 · Contextual embeddings come out of the encoder

In [ ]:
bert = AutoModel.from_pretrained(MODEL).to(device).eval()

cfg = bert.config
n_params = sum(p.numel() for p in bert.parameters())
print(f"layers={cfg.num_hidden_layers}  heads={cfg.num_attention_heads}  "
      f"hidden size={cfg.hidden_size}  parameters={n_params/1e6:.0f}M")

In [ ]:
with torch.no_grad():
    out = bert(**inputs.to(device))

h = out.last_hidden_state            # one vector per token
print(h.shape)                       # [batch, tokens, hidden] = [1, 5, 768]

tokens = tok.convert_ids_to_tokens(inputs["input_ids"][0])
for i, t in enumerate(tokens):
    print(f"h_{t:<6} -> first 5 of 768 numbers: {h[0, i, :5].cpu().numpy().round(3)}")

Those are the five vectors from the slide: **h_CLS, h_the, h_bat, h_flew, h_SEP**.

### Does "bat" really depend on its sentence?
The helper below finds a word in a sentence and returns its contextual vector. If the tokenizer split the word into pieces, it averages them.

In [ ]:
def word_vector(sentence, word, layer=-1):
    # Contextual vector for `word` in `sentence` (layer -1 = encoder output).
    enc = tok(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        hs = bert(**enc, output_hidden_states=True).hidden_states   # 13 tensors: embeddings + 12 layers
    ids = enc["input_ids"][0].tolist()
    piece_ids = tok(word, add_special_tokens=False)["input_ids"]
    for start in range(len(ids) - len(piece_ids) + 1):
        if ids[start:start + len(piece_ids)] == piece_ids:
            return hs[layer][0, start:start + len(piece_ids)].mean(dim=0)
    raise ValueError(f"'{word}' not found in: {tok.convert_ids_to_tokens(ids)}")

animal   = ["The bat flew out of the cave.",
            "A bat hung upside down in the barn.",
            "The bat ate insects at night."]
baseball = ["He swung the bat at the ball.",
            "She bought a new wooden bat for the game.",
            "The player gripped the bat tightly."]
sentences = animal + baseball

vecs = torch.stack([word_vector(s, "bat") for s in sentences])
sim = F.cosine_similarity(vecs.unsqueeze(1), vecs.unsqueeze(0), dim=-1).cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim, cmap="Blues", vmin=sim.min(), vmax=1)
labels = [s if len(s) < 32 else s[:30] + "…" for s in sentences]
ax.set_xticks(range(6), labels, rotation=45, ha="right"); ax.set_yticks(range(6), labels)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center",
                color="white" if sim[i, j] > (sim.min() + 1) / 2 else "black")
ax.set_title('Cosine similarity of h_bat across sentences')
plt.colorbar(im); plt.tight_layout(); plt.show()

**What to look for:** the two 3×3 blocks on the diagonal (animal–animal, baseball–baseball) should be more similar than the off-diagonal blocks (animal–baseball). Same word, same spelling, different vectors.

### Compare with the static embedding table
Before the encoder, BERT looks each token up in a table: `bert.embeddings.word_embeddings`. That lookup can't see context.

In [ ]:
bat_id = tok.convert_tokens_to_ids("bat")
static_bat = bert.embeddings.word_embeddings.weight[bat_id]
print("Static 'bat' vector shape:", static_bat.shape)
print("It is the same row of the table for every sentence above, so its cosine similarity is always 1.00")

### Watch context build up, layer by layer
`hidden_states[0]` is the input (table lookup + position), and `hidden_states[12]` is the encoder output. How different are the two senses of *bat* at each layer?

In [ ]:
a, b = "The bat flew out of the cave.", "He swung the bat at the ball."
sims = [F.cosine_similarity(word_vector(a, "bat", L), word_vector(b, "bat", L), dim=0).item()
        for L in range(cfg.num_hidden_layers + 1)]

plt.figure(figsize=(7, 3.5))
plt.plot(range(len(sims)), sims, marker="o")
plt.xlabel("Layer (0 = embedding table + position)"); plt.ylabel("cosine similarity")
plt.title('"bat" (animal) vs "bat" (baseball), layer by layer')
plt.grid(alpha=0.3); plt.show()

At layer 0 the two vectors differ only because *bat* sits at a different position in each sentence. As each self-attention layer mixes in the surrounding words, the vectors drift apart.

> **Question:** Why isn't the layer-0 similarity exactly 1.0?

## Part 4 · A real version of the slide's 2-D map
The lecture slide was illustrative. Here we compute it: contextual vectors for several words, squashed from 768 dimensions to 2 with PCA.

In [ ]:
from sklearn.decomposition import PCA

points = [(s, "bat") for s in sentences] + [
    ("The owl flew out of the barn.", "owl"),
    ("A bird landed on the fence.", "bird"),
    ("Insects buzzed around the cave.", "insects"),
    ("He hit the ball over the fence.", "ball"),
    ("She caught it in her glove.", "glove"),
    ("The pitcher threw a fast pitch.", "pitch"),
]
X = torch.stack([word_vector(s, w) for s, w in points]).cpu().numpy()
xy = PCA(n_components=2).fit_transform(X)

plt.figure(figsize=(8, 6))
for (s, w), (x, y) in zip(points, xy):
    is_bat = w == "bat"
    color = "tab:orange" if is_bat else "tab:gray"
    plt.scatter(x, y, s=90 if is_bat else 40, color=color)
    label = f"bat: {s[:24]}…" if is_bat else w
    plt.annotate(label, (x, y), xytext=(5, 4), textcoords="offset points", fontsize=9)
plt.title("Contextual vectors projected to 2-D (PCA)")
plt.xticks([]); plt.yticks([]); plt.show()

PCA throws away most of the 768 dimensions, so the picture is rough. Do the animal *bat*s sit closer to *owl*, *bird*, *insects* than to *ball*, *glove*, *pitch*?

## Part 5 · Add a small neural network for sentiment (frozen encoder)

This is the slide's diagram in code:

```
"The movie was great" ──► BERT (frozen) ──► h_CLS (768) ──► Linear 768→256 ──► ReLU ──► Linear 256→2 ──► softmax
```

Because BERT is **frozen**, we can run it once over every sentence, save the h_CLS vectors, and then train only the small head. That trains in seconds, even on a CPU.

### Load a labeled dataset from the Hub
**SST-2**: single sentences from movie reviews, labeled 0 = negative, 1 = positive.

In [ ]:
from datasets import load_dataset

sst2 = load_dataset("stanfordnlp/sst2")
print(sst2)
print(sst2["train"][0], sst2["train"][1])

In [ ]:
N_TRAIN = 4000                      # a subset keeps this fast; try more later
train = sst2["train"].shuffle(seed=0).select(range(N_TRAIN))
val   = sst2["validation"]          # 872 labeled sentences (the test split has no labels)
print(len(train), len(val), "| fraction positive in train:", np.mean(train["label"]).round(2))

### Step 1 · Run frozen BERT once and keep h_CLS

In [ ]:
@torch.no_grad()
def cls_features(texts, batch_size=64):
    feats = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], padding=True, truncation=True,
                  max_length=128, return_tensors="pt").to(device)
        h = bert(**enc).last_hidden_state        # [batch, tokens, 768]
        feats.append(h[:, 0].cpu())              # position 0 = [CLS]
    return torch.cat(feats)

X_train, y_train = cls_features(train["sentence"]), torch.tensor(train["label"])
X_val,   y_val   = cls_features(val["sentence"]),   torch.tensor(val["label"])
print(X_train.shape, X_val.shape)

### Step 2 · Define and train the head

In [ ]:
head = nn.Sequential(
    nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.1),
    nn.Linear(256, 2),
)
print(f"Head parameters: {sum(p.numel() for p in head.parameters()):,}  "
      f"(BERT has {n_params:,})")

opt = torch.optim.Adam(head.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()          # applies softmax internally

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

history = []
for epoch in range(30):
    head.train()
    perm = torch.randperm(len(X_train))
    for i in range(0, len(X_train), 64):
        idx = perm[i:i + 64]
        loss = loss_fn(head(X_train[idx]), y_train[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    history.append((accuracy(head, X_train, y_train), accuracy(head, X_val, y_val)))
    if epoch % 5 == 4:
        print(f"epoch {epoch+1:2d}  train acc {history[-1][0]:.3f}  val acc {history[-1][1]:.3f}")

plt.plot([h[0] for h in history], label="train"); plt.plot([h[1] for h in history], label="validation")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.grid(alpha=0.3); plt.show()

> **Question:** Train accuracy keeps climbing but validation levels off. What is happening, and what could you change?

### Step 3 · Try your own sentences

In [ ]:
def predict(texts, model=head):
    X = cls_features(texts)
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(X), dim=1)
    for t, p in zip(texts, probs):
        print(f"{p[1]:.2f} positive | {t}")

predict(["The movie was great",
         "The movie was a waste of two hours",
         "Not bad at all",
         "I expected to hate it, but I loved it"])

## Part 6 · Fine-tune everything (GPU recommended)

Now let gradients flow into BERT too. This is the `SentimentClassifier` from the lecture. Because BERT changes every step, we can't cache features; each batch goes through the full encoder.

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, model_name=MODEL):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.head = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 2))

    def forward(self, input_ids, attention_mask):
        h = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        cls = h[:, 0]                  # [batch, 768]
        return self.head(cls)          # logits [batch, 2]

In [ ]:
from torch.utils.data import DataLoader

n = N_TRAIN if device == "cuda" else 500          # keep CPU runs short
if device == "cpu":
    print("No GPU: training on only 500 examples. Switch to a T4 GPU for the real run.")

def collate(batch):
    enc = tok([b["sentence"] for b in batch], padding=True, truncation=True,
              max_length=128, return_tensors="pt")
    enc["labels"] = torch.tensor([b["label"] for b in batch])
    return enc

train_dl = DataLoader(train.select(range(n)), batch_size=32, shuffle=True, collate_fn=collate)
val_dl   = DataLoader(val, batch_size=64, collate_fn=collate)

model = SentimentClassifier().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)   # small LR: don't wreck pretrained weights

def evaluate(model):
    model.eval(); correct = 0
    with torch.no_grad():
        for b in val_dl:
            b = {k: v.to(device) for k, v in b.items()}
            correct += (model(b["input_ids"], b["attention_mask"]).argmax(1) == b["labels"]).sum().item()
    return correct / len(val)

for epoch in range(2):
    model.train()
    for step, b in enumerate(train_dl):
        b = {k: v.to(device) for k, v in b.items()}
        loss = loss_fn(model(b["input_ids"], b["attention_mask"]), b["labels"])
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 25 == 0:
            print(f"epoch {epoch+1} step {step:3d}  loss {loss.item():.3f}")
    print(f"epoch {epoch+1}  validation accuracy: {evaluate(model):.3f}")

> **Compare:** frozen encoder (Part 5) vs fine-tuned encoder (Part 6). Which is more accurate? Which was cheaper to train? When would you choose each?

## Part 7 · Same encoder, different heads

Each pipeline below is an encoder with a different head on top, downloaded from the Hub.

| Task | Which encoder outputs the head uses | Hub model |
|---|---|---|
| Fill in the blank | the `[MASK]` token's vector → a score for every word in the vocabulary | `bert-base-uncased` |
| Named entities | every token's vector → a tag per token | `dslim/bert-base-NER` |
| Question answering | every token's vector → start and end scores | `distilbert/distilbert-base-cased-distilled-squad` |
| Search / RAG | average of all token vectors → one sentence vector | `sentence-transformers/all-MiniLM-L6-v2` |

### Fill in the blank (how BERT was pretrained)

In [ ]:
fill = pipeline("fill-mask", model=MODEL)
for s in ["The [MASK] flew out of the cave.", "He swung the [MASK] at the ball."]:
    print(s, "->", [(r["token_str"], round(r["score"], 2)) for r in fill(s, top_k=5)])

### Named entities (one label per token)

In [ ]:
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
ner("Ada Lovelace worked with Charles Babbage in London on the Analytical Engine.")

### Question answering (find the answer span)

In [ ]:
qa = pipeline("question-answering", model="distilbert/distilbert-base-cased-distilled-squad")
context = ("Bats are the only mammals capable of true flight. Most bats are nocturnal "
           "and use echolocation to find insects in the dark.")
qa(question="How do bats find insects?", context=context)

### Sentence embeddings for search (the R in RAG)

In [ ]:
from sentence_transformers import SentenceTransformer

st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
docs = ["Bats use echolocation to hunt at night.",
        "A wooden baseball bat is usually made of maple or ash.",
        "The stock market fell sharply on Tuesday.",
        "Owls are nocturnal birds of prey."]
query = "Which animals hunt in the dark?"

doc_emb, q_emb = st.encode(docs, convert_to_tensor=True), st.encode(query, convert_to_tensor=True)
scores = F.cosine_similarity(q_emb.unsqueeze(0), doc_emb).cpu()
for score, d in sorted(zip(scores.tolist(), docs), reverse=True):
    print(f"{score:.2f}  {d}")

That ranking step is the *retrieval* in Retrieval-Augmented Generation: an **encoder** finds the relevant documents, then a **decoder** writes the answer.

## Part 8 · Exercises

1. **Another ambiguous word.** Repeat the Part 3 similarity matrix for *bank* (river vs money), *spring* (season vs coil), or *bass* (fish vs music). Paste your heatmap.
2. **Pooling.** In Part 5, replace `h[:, 0]` with the **mean** of all real token vectors (use the `attention_mask` so padding is ignored). Does validation accuracy go up or down?
3. **Head size.** Replace the two-layer head with a single `nn.Linear(768, 2)`. How much accuracy do you lose, if any?
4. **Swap the encoder.** Change `MODEL` to `distilbert/distilbert-base-uncased` and rerun Parts 2–5. Compare speed, parameter count, and accuracy.
5. **Read a model card.** Open the card for `dslim/bert-base-NER`. What data was it trained on, and what limitation does the card mention?